# 20b - Follow-up: symptom text on the app's current confidence, with six more factsheets

**This is a follow-up to notebook 20, added after notebooks 21 and 21c, and reported as such.**

**Why:** notebook 20 (24 Oct) chose TF-IDF, the farmer's crop only, weight 0.25, acting only
below the 0.8 threshold. Notebook 21 then changed the confidence the app shows (rescaled within
the crop, T = 1.7002) and the threshold (0.9557, the same policy on the new scale), so "below
0.8" no longer means what it meant. Six more Plantwise factsheets for farmers (CC BY-SA 4.0)
were also found on 1 Nov 2026, mostly on early and late blight, the pair notebook 21c showed
the model confuses most. This notebook repeats notebook 20's comparison on the current
confidence, with the larger set of sentences. Nothing about the photos or the image model changes.

**What changes from notebook 20 (decided 1 Nov 2026, before this notebook was first run):**

| | Notebook 20 | Notebook 20b |
|---|---|---|
| Image confidence | 11-class probability of the crop's best class, T = 2.4381 | Rescaled within the crop, T = 1.7002 (notebook 21) |
| Answer or refer | confidence >= 0.8 | photo alone: crop share >= 0.5 and confidence >= 0.9557 |
| When the text acts | always, or below 0.8 | always, or **only on photos the app refers** |
| Sentences for choosing | 64 | 82 (64 + 18 from the new factsheets) |
| Locked test sentences | 8 | 16 (8 + 8) |

**Safety condition (fixed before running):** whether the farmer gets an answer is decided by
the photo alone. The text can only reorder the diseases on photos that are referred, so it can
never turn a referral into an answer or change an answer the farmer sees; it changes the first
guess the officer sees. Options where the text acts "always" are measured and reported but
cannot be chosen.

**The new factsheets and their split (fixed before running):** a factsheet sharing an author
with one already used goes to that one's split; of the rest, one per disease goes to the locked
test set, picked with the project seed; 20147801027 is left out because it repeats held-out
20137803397 word for word. Sentences were kept or dropped by notebook 11's rule (a visible symptom
is kept; background, cause, yield and other crops are dropped).

**Rule (notebook 20's, unchanged except for the safety condition):**
1. Keep only options with harm <= 2% and, with wrong text, accuracy of answered photos of at
   least 90%.
2. Of those, the largest help, if its paired bootstrap range lies entirely above zero.
3. Tied options (help at least the best's lower bound) go to the smaller encoder, then the
   lower weight, then "farmer's crop" over "all 8".
4. If nothing passes with the 82 sentences, notebook 20's winner (TF-IDF, crop, 0.25) is checked
   with its own 64 sentences on the current confidence; if it passes the same rule it stays, and
   both runs are reported.
5. Otherwise the photo alone stays and the text stays an aid for the officer.

**Reported only:** text-only accuracy; the locked test sentences on the test set and PlantDoc;
help and harm per class; the first guess on referred photos.

**Limitations stated up front:** factsheet sentences, not farmers' words; northern leaf blight,
common rust and potato early blight still have no test sentence (no openly licensed farmer
factsheet was found for them).

Runs on CPU; about 20-30 minutes, mostly downloading the five language models.

In [1]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Mounted at /content/drive


In [2]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
Refresh index: 100% (501/501), done.
## main...origin/main
 M ml/notebooks/20b_symptom_text_recheck.ipynb
?? ml/results/fusion_quick_test/


In [3]:
%pip install -q -r {ML_DIR}/requirements.txt

## Settings, the app's current confidence and the sentences

In [4]:
import importlib
import json
import shutil

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer

from src import calibration as cal, fusion, metrics, text_encoders, training
for module in (cal, fusion, metrics, text_encoders, training):
    importlib.reload(module)

SEED = config.RANDOM_SEED
HARM_LIMIT, ANSWERED_TARGET = 0.02, 0.90
WEIGHTS = (0.25, 0.5, 0.75, 1.0)
SCOPES = ("crop", "all8")
GATES = ("always", "referred only")
ELIGIBLE_GATE = "referred only"      # the safety condition: text acts only on photos the app refers
ENCODERS = list(text_encoders.ENCODERS)
SIZE = {"tfidf": 0, "minilm": 22, "distilbert": 66, "bert_base": 110, "mpnet": 110, "scibert": 110}  # million weights
PAIRINGS, FOLDS, BOOTSTRAP, CAL_FOLDS = 5, 4, 1000, 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Added on 1 Nov 2026 (ml/data/symptoms/sources.csv); every other kept sentence is notebook 20's.
NEW_FACTSHEETS = {"20137804212", "20127802679", "20147801362", "20127801047", "20157800166", "20147801161"}
# Notebook 20's winner, its "below 0.8" translated to "only on photos the app refers".
OLD_WINNER = {"encoder": "tfidf", "scope": "crop", "weight": 0.25, "gate": "referred only"}

health = json.loads((ML_DIR / "results" / "health_check" / "decision.json").read_text())
assert health["A3"]["adopted"], "Notebook 21's rescaling is not deployed; this notebook assumes it is."
MEMBERS = health["deployed"]["model_runs"]
TEMPERATURE = health["deployed"]["temperature"]
CROP_TEMPERATURE = health["A3"]["temperature_within_crop"]
THRESHOLD = health["A3"]["threshold"]
FOLDER = {"d17": config.MODELS_DIR / "d17_comparison", "d22": config.MODELS_DIR / "d22_comparison"}
OUT_DIR = config.MODELS_DIR / "fusion_d20b"
REPO_RESULTS = ML_DIR / "results" / "fusion_d20b"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_RESULTS.mkdir(parents=True, exist_ok=True)

SETS = ["selection", "test", "ood_test"]
CLASSES = list(config.PLANTVILLAGE_CLASSES)
INDEX = {name: i for i, name in enumerate(CLASSES)}
CLASS_CROPS = np.array([{"Corn_(maize)": "maize", "Tomato": "tomato", "Potato": "potato"}[c.split("___")[0]]
                        for c in CLASSES])
SHORT = {c: c.replace("Corn_(maize)___", "Maize ").replace("___", " ").replace("_", " ").strip() for c in CLASSES}
DISEASE = np.flatnonzero(~fusion.healthy_mask(CLASSES))            # the 8 disease classes, in text-classifier order
HEALTHY = fusion.healthy_mask(CLASSES)
FIELD_SOURCES = ["fieldplant", "tanzania"]
NEG = -1e4  # far below any real score: the other crops' classes can never be chosen


def within_crop(scores, crops):
    keep = CLASS_CROPS[None, :] == np.asarray(crops)[:, None]
    return np.where(keep, scores, NEG)


PARTS = {}
for name in SETS:
    rows = pd.read_csv(config.SPLITS_DIR / f"{name}.csv")
    labels = rows["class"].map(INDEX)
    keep = labels.notna().to_numpy()
    lab = labels[keep].astype(int).to_numpy()
    members = [np.load(FOLDER[m[:3]] / f"{m}__{name}.npy")[keep] for m in MEMBERS]
    raw = np.log(np.mean([metrics.softmax(s) for s in members], axis=0) + 1e-12)   # the backend's combination
    PARTS[name] = {"rows": rows[keep].reset_index(drop=True), "labels": lab, "crops": CLASS_CROPS[lab], "raw": raw}

# The app's confidence (notebook 21): the farmer's crop's classes only, at T = 1.7002; the crop guard
# uses the 11-class probability at T = 2.4381. Both temperatures were fitted on the selection set,
# so there each photo is scored by temperatures fitted without it, as in notebook 21.
for name, part in PARTS.items():
    if name == "selection":
        eleven = cal.out_of_fold("temperature", part["raw"], part["labels"], CAL_FOLDS, SEED)
        part["probs"] = cal.out_of_fold("temperature", within_crop(part["raw"], part["crops"]), part["labels"],
                                        CAL_FOLDS, SEED)
    else:
        eleven = cal.Calibrator("temperature", {"temperature": TEMPERATURE}).probabilities(part["raw"])
        part["probs"] = cal.Calibrator("temperature", {"temperature": CROP_TEMPERATURE}).probabilities(
            within_crop(part["raw"], part["crops"]))
    share = (eleven * (CLASS_CROPS[None, :] == part["crops"][:, None])).sum(axis=1)
    part["referred"] = (share < 0.5) | (part["probs"].max(axis=1) < THRESHOLD)
FIELD_TEST = PARTS["test"]["rows"]["source"].isin(FIELD_SOURCES).to_numpy()
sel = PARTS["selection"]


def app(probs, crops, labels):
    conf, predicted = cal.crop_confidence(probs, CLASS_CROPS, crops)
    return conf, predicted == labels


# Check: the photo alone must give notebook 21's numbers for the deployed setting (a3_rescaling.csv).
EXPECTED = {"selection": (0.24339, 0.99785), "test": (0.92339, 0.99911), "ood_test": (0.17209, 0.67073)}
for name, (answered, accuracy) in EXPECTED.items():
    part = PARTS[name]
    ok = app(part["probs"], part["crops"], part["labels"])[1]
    a = ~part["referred"]
    print(f"{name}: answered {a.mean():.4f} (notebook 21: {answered}), accuracy of answered {ok[a].mean():.4f} ({accuracy})")
    assert abs(a.mean() - answered) < 1e-3 and abs(ok[a].mean() - accuracy) < 1e-3, "does not match notebook 21"

descriptions = pd.read_csv(ML_DIR / "data" / "symptoms" / "descriptions.csv", encoding="utf-8-sig",
                           dtype={"factsheet_id": str})
descriptions = descriptions[descriptions["keep"] == "y"].reset_index(drop=True)
descriptions["label"] = descriptions["class_name"].map(INDEX)
descriptions["disease"] = descriptions["label"].map({c: i for i, c in enumerate(DISEASE)})


def make_pool(frame):
    """Train and val sentences in FOLDS folds stratified by disease, exactly as notebook 20 made them."""
    pool = frame[frame["split"].isin(["train", "val"])].reset_index(drop=True)
    rng = np.random.default_rng(SEED)
    pool["fold"] = -1
    for _, group in pool.groupby("disease"):
        order = rng.permutation(group.index.to_numpy())
        pool.loc[order, "fold"] = np.arange(len(order)) % FOLDS
    return pool


pool = make_pool(descriptions)
old_pool = make_pool(descriptions[~descriptions["factsheet_id"].isin(NEW_FACTSHEETS)])
locked = descriptions[descriptions["split"] == "test"].reset_index(drop=True)
assert len(old_pool) == 64, f"notebook 20 chose with 64 sentences, found {len(old_pool)}"
print(f"Sentences: {len(pool)} for choosing ({len(old_pool)} from notebook 20), {len(locked)} locked for the test")
display(pd.DataFrame({"pool now": pool["class_name"].map(SHORT).value_counts(),
                      "pool in notebook 20": old_pool["class_name"].map(SHORT).value_counts(),
                      "locked test": locked["class_name"].map(SHORT).value_counts()}).fillna(0).astype(int))

selection: answered 0.2434 (notebook 21: 0.24339), accuracy of answered 0.9978 (0.99785)
test: answered 0.9234 (notebook 21: 0.92339), accuracy of answered 0.9991 (0.99911)
ood_test: answered 0.1721 (notebook 21: 0.17209), accuracy of answered 0.6707 (0.67073)
Sentences: 82 for choosing (64 from notebook 20), 16 locked for the test


,pool now,pool in notebook 20,locked test
class_name,,,
Maize Cercospora leaf spot Gray leaf spot,5,5,1
Maize Common rust,5,5,0
Maize Northern Leaf Blight,4,4,0
Potato Early blight,4,4,0
Potato Late blight,26,19,3
Tomato Bacterial spot,9,9,2
Tomato Early blight,17,11,3
Tomato Late blight,12,7,7


## Text classifiers: out-of-fold scores for the sentences used to choose, and the final ones

As in notebook 20: one linear layer over the 8 diseases on frozen sentence vectors; TF-IDF's
vocabulary is refitted inside each fold. TF-IDF is also trained on notebook 20's 64 sentences,
for rule 4.

In [5]:
def text_classifier(vectors, diseases, seed=SEED):
    bank = fusion.TextBank({int(DISEASE[d]): vectors[diseases == d] for d in np.unique(diseases)}, vectors.shape[1])
    return fusion.train_head("late", None, None, bank, None, CLASSES, 0.0, seed, DEVICE)


@torch.no_grad()
def log_probs(head, vectors):
    return torch.log_softmax(head(torch.as_tensor(vectors, dtype=torch.float32, device=DEVICE)), 1).cpu().numpy()


def text_scores(pool, encoders):
    """encoder -> {"pool": out-of-fold log-probs, "test": the final classifier on the locked sentences}."""
    texts, locked_texts = pool["text"].tolist(), locked["text"].tolist()
    scores, heads = {}, {}
    for encoder in encoders:
        oof = np.zeros((len(pool), len(DISEASE)))
        if encoder != "tfidf":
            frozen = text_encoders.encode(encoder, texts + locked_texts, texts, DEVICE)
        for k in range(FOLDS):
            fit_on, held = (pool["fold"] != k).to_numpy(), (pool["fold"] == k).to_numpy()
            if encoder == "tfidf":
                vectors = text_encoders.encode("tfidf", texts, [t for t, f in zip(texts, fit_on) if f])
            else:
                vectors = frozen[:len(pool)]
            head = text_classifier(vectors[fit_on], pool["disease"].to_numpy()[fit_on])
            oof[held] = log_probs(head, vectors[held])
        everything = text_encoders.encode("tfidf", texts + locked_texts, texts) if encoder == "tfidf" else frozen
        heads[encoder] = text_classifier(everything[:len(pool)], pool["disease"].to_numpy())
        scores[encoder] = {"pool": oof, "test": log_probs(heads[encoder], everything[len(pool):])}
        print(f"{encoder}: done")
    return scores, heads


TEXT, HEADS = text_scores(pool, ENCODERS)
TEXT_OLD, HEADS_OLD = text_scores(old_pool, ["tfidf"])


def text_only(lp, sentences):
    truth = sentences["disease"].to_numpy()
    crops = CLASS_CROPS[DISEASE][truth]
    in_crop = np.where(CLASS_CROPS[DISEASE][None, :] == crops[:, None], lp, -np.inf)
    return (lp.argmax(1) == truth).mean(), (in_crop.argmax(1) == truth).mean()


text_rows = []
for label, scores, sentences in [*[(e, TEXT[e], pool) for e in ENCODERS], ("tfidf, notebook 20's 64", TEXT_OLD["tfidf"], old_pool)]:
    a8, ac = text_only(scores["pool"], sentences)
    t8, tc = text_only(scores["test"], locked)
    text_rows.append({"encoder": label, "pool out-of-fold: 8-way": a8, "pool out-of-fold: crop given": ac,
                      "test: 8-way": t8, "test: crop given": tc})
text_table = pd.DataFrame(text_rows).set_index("encoder")
print("Text alone, share of sentences whose disease is named correctly (reported only):")
display(text_table.round(3))
# Notebook 20 reported 0.328 (8-way) and 0.641 (crop given) for TF-IDF on its 64 sentences.
print("TF-IDF on notebook 20's 64 sentences reproduces notebook 20:",
      bool(np.allclose(text_table.loc["tfidf, notebook 20's 64", ["pool out-of-fold: 8-way", "pool out-of-fold: crop given"]],
                       [0.328125, 0.640625])))

tfidf: done


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

... (38 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

scibert: done
tfidf: done
Text alone, share of sentences whose disease is named correctly (reported only):


,pool out-of-fold: 8-way,pool out-of-fold: crop given,test: 8-way,test: crop given
encoder,,,,
tfidf,0.354,0.720,0.250,0.688
distilbert,0.390,0.671,0.312,0.500
bert_base,0.378,0.659,0.375,0.562
minilm,0.390,0.683,0.375,0.625
mpnet,0.402,0.671,0.375,0.562
scibert,0.366,0.744,0.438,0.750
"tfidf, notebook 20's 64",0.328,0.641,0.250,0.500


TF-IDF on notebook 20's 64 sentences reproduces notebook 20: True


## Pairing photos with sentences, and the safety condition

As in notebook 20: each diseased photo gets a random sentence of its own disease (right text)
and, separately, a random sentence of another disease of its crop (wrong text); healthy photos
get no right text and, as wrong text, a sentence of any disease of their crop. Five pairings
per photo. Whether a photo is answered is decided before the text, by the photo alone.

In [6]:
def pairings(labels, sentences, wrong, seed):
    # Index of the paired sentence for each photo, -1 for none.
    rng = np.random.default_rng(seed)
    by_class = {c: g.index.to_numpy() for c, g in sentences.groupby("label")}
    sentence_crop = CLASS_CROPS[sentences["label"].to_numpy()]
    chosen = np.full(len(labels), -1)
    for i, label in enumerate(labels):
        if not wrong:
            options = by_class.get(int(label), [])
        else:
            options = np.flatnonzero((sentence_crop == CLASS_CROPS[label]) & (sentences["label"].to_numpy() != label))
        if len(options):
            chosen[i] = options[rng.integers(len(options))]
    return chosen


def fused(probs, crops, text_lp, chosen, weight, scope):
    has = (chosen >= 0).astype(float)
    lp = np.where(has[:, None] > 0, text_lp[np.maximum(chosen, 0)], 0.0)
    return fusion.crop_fusion_probabilities(probs, lp, has, DISEASE, CLASS_CROPS, crops, weight,
                                            within_crop=scope == "crop")


def gated(probs, mixed, referred, gate):
    """With "referred only" the text changes nothing on a photo the app answers (the safety condition)."""
    return mixed if gate == "always" else np.where(referred[:, None], mixed, probs)


def build_pairs(choose_from):
    return {name: {kind: [pairings(PARTS[name]["labels"], choose_from if name == "selection" else locked,
                                   kind == "wrong", SEED + r) for r in range(PAIRINGS)]
                   for kind in ("right", "wrong")} for name in SETS}


PAIRS = {"new": build_pairs(pool), "old": build_pairs(old_pool)}
print("Selection photos with right text:", f"{(PAIRS['new']['selection']['right'][0] >= 0).mean():.1%}",
      "| with wrong text:", f"{(PAIRS['new']['selection']['wrong'][0] >= 0).mean():.1%}",
      "| referred by the photo alone:", f"{sel['referred'].mean():.1%}")

Selection photos with right text: 57.8% | with wrong text: 100.0% | referred by the photo alone: 75.7%


## All 96 options on the selection set

In [7]:
def evaluate(name, encoder, weight, scope, gate, mask=None, sentences="new"):
    part = PARTS[name]
    m = np.ones(len(part["labels"]), bool) if mask is None else mask
    probs, labels, crops, referred = part["probs"][m], part["labels"][m], part["crops"][m], part["referred"][m]
    answered = ~referred                 # decided by the photo alone, whatever the text says
    ok0 = app(probs, crops, labels)[1]
    text_lp = (TEXT if sentences == "new" else TEXT_OLD)[encoder]["pool" if name == "selection" else "test"]
    out = {}
    for kind in ("right", "wrong"):
        oks = np.array([app(gated(probs, fused(probs, crops, text_lp, chosen[m], weight, scope), referred, gate),
                            crops, labels)[1] for chosen in PAIRS[sentences][name][kind]])
        out[kind] = {"ok": oks.mean(axis=0),
                     "accuracy of answered": oks[:, answered].mean() if answered.any() else np.nan}
    healthy = HEALTHY[labels]
    wrong_share = 1 - out["wrong"]["ok"]
    return {"photos": int(m.sum()), "accuracy image": ok0.mean(), "accuracy right text": out["right"]["ok"].mean(),
            "help": out["right"]["ok"].mean() - ok0.mean(),
            "harm": wrong_share[ok0].mean() if ok0.any() else np.nan,
            "harm (diseased photos)": wrong_share[ok0 & ~healthy].mean() if (ok0 & ~healthy).any() else np.nan,
            "harm (healthy photos)": wrong_share[ok0 & healthy].mean() if (ok0 & healthy).any() else np.nan,
            "answered": answered.mean(),
            "accuracy of answered image": ok0[answered].mean() if answered.any() else np.nan,
            "accuracy of answered right text": out["right"]["accuracy of answered"],
            "accuracy of answered wrong text": out["wrong"]["accuracy of answered"],
            "first guess on referred, image": ok0[referred].mean() if referred.any() else np.nan,
            "first guess on referred, right text": out["right"]["ok"][referred].mean() if referred.any() else np.nan,
            "_diff": out["right"]["ok"] - ok0}


resample = np.random.default_rng(SEED).integers(0, len(sel["labels"]), (BOOTSTRAP, len(sel["labels"])))  # paired: same photos for all


def scored(option, sentences="new"):
    r = evaluate("selection", **option, sentences=sentences)
    low, high = np.percentile(r.pop("_diff")[resample].mean(axis=1), [2.5, 97.5])
    return {**option, **r, "help low": low, "help high": high}


options = pd.DataFrame([scored({"encoder": e, "scope": s, "weight": w, "gate": g})
                        for e in ENCODERS for s in SCOPES for w in WEIGHTS for g in GATES])
print(len(options), "options evaluated")

96 options evaluated


## The rule

In [8]:
def passes(row):
    safe = row["harm"] <= HARM_LIMIT and (pd.isna(row["accuracy of answered wrong text"])
                                         or row["accuracy of answered wrong text"] >= ANSWERED_TARGET)
    return bool(safe), bool(row["help low"] > 0)


checks = [passes(r) for _, r in options.iterrows()]
options["safe"], options["helps"] = [s for s, _ in checks], [h for _, h in checks]
options["eligible"] = options["gate"] == ELIGIBLE_GATE
options["passes"] = options["safe"] & options["helps"] & options["eligible"]
options["order"] = list(zip(options["encoder"].map(SIZE), options["encoder"], options["weight"], options["scope"] != "crop"))

old_check = scored(OLD_WINNER, sentences="old")
old_check["safe"], old_check["helps"] = passes(old_check)
old_check["passes"] = old_check["safe"] and old_check["helps"]

passing = options[options["passes"]]
if len(passing):
    best = passing.loc[passing["help"].idxmax()]
    tied = passing[passing["help"] >= best["help low"]]
    winner = tied.loc[min(tied.index, key=lambda i: tied.at[i, "order"])]
    WINNER = {k: winner[k] for k in ("encoder", "scope", "gate")} | {"weight": float(winner["weight"])}
    SENTENCES_USED = "new"
    VERDICT = (f"Rule 1-3, {len(pool)} sentences: text is used: {WINNER} (help {winner['help']:+.4f}, range "
               f"{winner['help low']:+.4f} to {winner['help high']:+.4f}; harm {winner['harm']:.4f}; "
               f"{len(tied)} option(s) tied with the best)")
elif old_check["passes"]:
    winner = pd.Series(old_check)
    WINNER, SENTENCES_USED = dict(OLD_WINNER), "old"
    VERDICT = (f"Rule 4: nothing passes with {len(pool)} sentences; notebook 20's winner passes with its 64 on the "
               f"current confidence (help {old_check['help']:+.4f}, range {old_check['help low']:+.4f} to "
               f"{old_check['help high']:+.4f}; harm {old_check['harm']:.4f}) and stays")
else:
    winner, WINNER, SENTENCES_USED = None, None, None
    VERDICT = (f"Rule 5: nothing passes ({int((options['safe'] & options['eligible']).sum())} safe, "
               f"{int((options['helps'] & options['eligible']).sum())} help beyond the range, none both; notebook 20's "
               "winner does not pass either): the photo alone stays and the text stays an aid for the officer")
print(VERDICT)

view = ["encoder", "scope", "weight", "gate", "help", "help low", "help high", "harm", "harm (healthy photos)",
        "accuracy of answered wrong text", "first guess on referred, image", "first guess on referred, right text", "passes"]
print("\nEvery option, largest help first:")
display(options.sort_values("help", ascending=False)[view].round(4).to_string(index=False))
print("\nNotebook 20's winner, with its 64 sentences, on the current confidence:")
display(pd.Series(old_check)[[v for v in view if v in old_check]].to_frame().T.round(4).to_string(index=False))
print("\nPhoto alone on the selection set:",
      options.iloc[0][["accuracy image", "answered", "accuracy of answered image"]].round(4).to_dict())

Rule 1-3, 82 sentences: text is used: {'encoder': 'tfidf', 'scope': 'crop', 'gate': 'referred only', 'weight': 0.25} (help +0.0043, range +0.0021 to +0.0064; harm 0.0071; 4 option(s) tied with the best)

Every option, largest help first:


'   encoder scope  weight          gate   help  help low  help high   harm  harm (healthy photos)  accuracy of answered wrong text  first guess on referred, image  first guess on referred, right text  passes\n    minilm  crop    1.00        always 0.0292    0.0221     0.0367 0.1748                 0.0228                           0.9966                          0.6648                               0.7034   False\n    minilm  crop    1.00 referred only 0.0292    0.0221     0.0367 0.1744                 0.0228                           0.9978                          0.6648                               0.7034   False\n    minilm  all8    1.00 referred only 0.0292    0.0221     0.0367 0.1744                 0.0228                           0.9978                          0.6648                               0.7034   False\n    minilm  all8    1.00        always 0.0292    0.0221     0.0367 0.1748                 0.0228                           0.9966                          0.6648      


Notebook 20's winner, with its 64 sentences, on the current confidence:


'encoder scope weight          gate      help  help low help high      harm harm (healthy photos) accuracy of answered wrong text first guess on referred, image first guess on referred, right text passes\n  tfidf  crop   0.25 referred only  0.005339  0.003088  0.007747  0.006807              0.000148                        0.997849                       0.664822                            0.671878   True'


Photo alone on the selection set: {'accuracy image': 0.7458780423972782, 'answered': 0.24339178225595393, 'accuracy of answered image': 0.9978494623655914}


## Reported only: the locked test sentences, PlantDoc, and per class

In [9]:
CHECKED = WINNER or OLD_WINNER       # what the report shows when nothing passes: notebook 20's winner
CHECKED_SENTENCES = SENTENCES_USED or "old"
safe = options[options["safe"] & options["eligible"]]
REPORTED = {"chosen" if WINNER else "notebook 20's winner (not chosen)": (CHECKED, CHECKED_SENTENCES)}
for encoder in ENCODERS:
    pick = safe[safe["encoder"] == encoder]
    if len(pick):
        b = pick.loc[pick["help"].idxmax()]
        REPORTED[f"best safe {encoder}"] = ({k: b[k] for k in ("encoder", "scope", "gate")} | {"weight": float(b["weight"])}, "new")

report = []
for label, name, mask in [("test", "test", None), ("test: field photos", "test", FIELD_TEST), ("PlantDoc", "ood_test", None)]:
    for option_name, (option, sentences) in REPORTED.items():
        r = evaluate(name, **option, mask=mask, sentences=sentences)
        r.pop("_diff")
        report.append({"set": label, "option": option_name, **option, "sentences": sentences, **r})
report_table = pd.DataFrame(report)
display(report_table.round(4).to_string(index=False))
print("Classes with no locked test sentence (their photos get no text above):",
      sorted(SHORT[CLASSES[c]] for c in DISEASE if c not in set(locked["label"])))

per_class = []
for c in np.unique(sel["labels"]):
    m = sel["labels"] == c
    r = evaluate("selection", **CHECKED, mask=m, sentences=CHECKED_SENTENCES)
    r.pop("_diff")
    per_class.append({"class": SHORT[CLASSES[c]], **{k: r[k] for k in ("photos", "accuracy image", "accuracy right text",
                      "harm", "answered", "first guess on referred, image", "first guess on referred, right text")}})
per_class_table = pd.DataFrame(per_class).set_index("class")
print("Selection set per class, option:", CHECKED, "| sentences:", CHECKED_SENTENCES)
display(per_class_table.round(4))

'               set          option encoder scope          gate  weight sentences  photos  accuracy image  accuracy right text   help   harm  harm (diseased photos)  harm (healthy photos)  answered  accuracy of answered image  accuracy of answered right text  accuracy of answered wrong text  first guess on referred, image  first guess on referred, right text\n              test          chosen   tfidf  crop referred only    0.25       new    2441          0.9951               0.9953 0.0002 0.0000                  0.0000                 0.0000    0.9234                      0.9991                           0.9991                           0.9991                          0.9465                               0.9497\n              test best safe tfidf   tfidf  crop referred only    0.50       new    2441          0.9951               0.9954 0.0003 0.0001                  0.0001                 0.0000    0.9234                      0.9991                           0.9991                    

Classes with no locked test sentence (their photos get no text above): ['Maize Common rust', 'Maize Northern Leaf Blight', 'Potato Early blight']
Selection set per class, option: {'encoder': 'tfidf', 'scope': 'crop', 'gate': 'referred only', 'weight': 0.25} | sentences: new


,photos,accuracy image,accuracy right text,harm,answered,"first guess on referred, image","first guess on referred, right text"
class,,,,,,,
Maize Cercospora leaf spot Gray leaf spot,1380,0.7500,0.7565,0.0083,0.0572,0.7364,0.7433
Maize Common rust,126,0.9127,0.9175,0.0104,0.0079,0.9120,0.9168
Maize healthy,961,0.9469,0.9469,0.0000,0.5307,0.8869,0.8869
Tomato Bacterial spot,97,0.2474,0.2206,0.1250,0.0000,0.2474,0.2206
Tomato Early blight,447,0.4855,0.5043,0.0212,0.0022,0.4843,0.5031
Tomato Late blight,92,0.4565,0.4674,0.0524,0.0000,0.4565,0.4674
Tomato healthy,307,0.3192,0.3192,0.0061,0.0000,0.3192,0.3192
Potato Late blight,67,1.0000,1.0000,0.0000,0.8955,1.0000,1.0000
Potato healthy,344,0.9942,0.9942,0.0000,0.8110,0.9692,0.9692


## Save the decision

In [10]:
decision = {"verdict": VERDICT, "winner": WINNER, "sentences_used": SENTENCES_USED, "post_hoc_followup": True,
            "rule": {"harm_limit": HARM_LIMIT, "answered_target_with_wrong_text": ANSWERED_TARGET,
                     "eligible_gate": ELIGIBLE_GATE,
                     "help": "largest help whose paired bootstrap range lies above zero",
                     "tie": "help at least the best's lower bound; then smaller encoder, lower weight, farmer's crop",
                     "fallback": "notebook 20's winner with its 64 sentences, if it passes the same rule; else the photo alone",
                     "written": "2026-11-01, before the first run"},
            "image_model": {"members": MEMBERS, "temperature": TEMPERATURE, "crop_temperature": CROP_TEMPERATURE,
                            "threshold": THRESHOLD},
            "sentences": {"pool": len(pool), "pool_notebook_20": len(old_pool), "folds": FOLDS, "locked_test": len(locked),
                          "new_factsheets": sorted(NEW_FACTSHEETS)},
            "pairings_per_photo": PAIRINGS, "selection_photos": int(len(sel["labels"])),
            "selection_referred_by_photo": float(sel["referred"].mean()),
            "options_safe": int((options["safe"] & options["eligible"]).sum()),
            "options_helping": int((options["helps"] & options["eligible"]).sum()),
            "options_passing": int(options["passes"].sum()),
            "notebook_20_winner_check": {k: (v if not isinstance(v, (np.floating, np.bool_)) else v.item())
                                         for k, v in old_check.items()}}
training.save_json(decision, OUT_DIR / "decision.json")
options.drop(columns="order").to_csv(OUT_DIR / "options.csv", index=False)
text_table.to_csv(OUT_DIR / "text_only.csv")
report_table.to_csv(OUT_DIR / "report_sets.csv", index=False)
per_class_table.to_csv(OUT_DIR / "per_class.csv")

if WINNER and WINNER["encoder"] == "tfidf":
    # Everything the backend needs to run the chosen TF-IDF classifier without scikit-learn or torch,
    # plus check sentences with the log-probabilities it must reproduce.
    used = pool if SENTENCES_USED == "new" else old_pool
    head = (HEADS if SENTENCES_USED == "new" else HEADS_OLD)["tfidf"]
    texts = used["text"].tolist()
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(texts)
    assert np.allclose(vectorizer.transform(texts).toarray(), text_encoders.encode("tfidf", texts, texts), atol=1e-6)
    linear = head.net[1]
    examples = locked["text"].tolist()
    expected = log_probs(head, vectorizer.transform(examples).toarray().astype(np.float32))
    text_model = {"encoder": "tfidf", "lowercase": True, "token_pattern": vectorizer.token_pattern,
                  "ngram_range": [1, 2], "sublinear_tf": True, "norm": "l2",
                  "vocabulary": {term: int(i) for term, i in vectorizer.vocabulary_.items()},
                  "idf": vectorizer.idf_.tolist(), "diseases": [CLASSES[c] for c in DISEASE],
                  "weight": linear.weight.detach().cpu().numpy().tolist(),
                  "bias": linear.bias.detach().cpu().numpy().tolist(),
                  "fusion": WINNER, "sentences": len(texts), "source": "notebook 20b",
                  "checks": [{"text": t, "log_probs": lp.tolist()} for t, lp in zip(examples, expected)]}
    (OUT_DIR / "text_model.json").write_text(json.dumps(text_model))
    print("Saved text_model.json:", len(text_model["vocabulary"]), "terms,", len(texts), "sentences")
elif WINNER:
    torch.save((HEADS if SENTENCES_USED == "new" else HEADS_OLD)[WINNER["encoder"]].state_dict(), OUT_DIR / "text_head.pt")
    print("A language model won: its classifier is saved to Drive as text_head.pt; the backend needs that model too.")
print(json.dumps(decision, indent=1, default=float))

Saved text_model.json: 1106 terms, 82 sentences
{
 "verdict": "Rule 1-3, 82 sentences: text is used: {'encoder': 'tfidf', 'scope': 'crop', 'gate': 'referred only', 'weight': 0.25} (help +0.0043, range +0.0021 to +0.0064; harm 0.0071; 4 option(s) tied with the best)",
 "winner": {
  "encoder": "tfidf",
  "scope": "crop",
  "gate": "referred only",
  "weight": 0.25
 },
 "sentences_used": "new",
 "post_hoc_followup": true,
 "rule": {
  "harm_limit": 0.02,
  "answered_target_with_wrong_text": 0.9,
  "eligible_gate": "referred only",
  "help": "largest help whose paired bootstrap range lies above zero",
  "tie": "help at least the best's lower bound; then smaller encoder, lower weight, farmer's crop",
  "fallback": "notebook 20's winner with its 64 sentences, if it passes the same rule; else the photo alone",
  "written": "2026-11-01, before the first run"
 },
 "image_model": {
  "members": [
   "d17_resnet50_seed44",
   "d22_convnext_tiny_lr1e-04",
   "d22_efficientnet_b0"
  ],
  "temperat

## Copy the results into the repository, sync Drive, save to GitHub

Save the notebook first (Ctrl+S).

In [11]:
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json"}:
        shutil.copy(file, REPO_RESULTS / file.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

['decision.json', 'options.csv', 'per_class.csv', 'report_sets.csv', 'text_model.json', 'text_only.csv']
Mounted at /content/drive


In [12]:
from google.colab import userdata

from src import github_sync

REPO_RESULTS.mkdir(parents=True, exist_ok=True)
shutil.copy(ML_DIR / "notebooks" / "20b_symptom_text_recheck.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/symptom-text-20b",
    message="Symptom text follow-up: the current confidence and six more factsheets",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))

TimeoutException: Requesting secret GITHUB_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.